# Apertus-8B-BioInstruct × Genomeer Agent — Test Suite

> **Model** : `swiss-ai/Apertus-8B-Instruct-2509` + LoRA adapter `v2v4_r64`  
> **Adapter** : `/mnt/nfs/llmhub/Genomeer/sft/output/Models/v2v4_r64/Apertus-8B-BioInstruct`  
> **Kernel** : sft_env (transformers + peft + torch + CUDA)  

---

| # | Level | Description |
|---|-------|-------------|
| 1 | Simple Q&A | Définitions bioinformatiques — pas de code |
| 2 | Simple Q&A | Questions avancées — métagénomique + assemblage |
| 3 | Code simple | Parse FASTA, GC content, longueur |
| 4 | Code medium | K-mer analysis + stats table |
| 5 | Tool use | Téléchargement NCBI + stats génomiques |
| 6 | Pipeline | QC → Assembly → Stats comparison (multi-step) |


## CELLULE 0 — Installation des dépendances manquantes

In [1]:
import subprocess, sys

# Packages needed for the agent (not in sft_env by default)
_to_install = [
    "fastapi",
    "uvicorn[standard]",
    "langchain==0.3.25",
    "langgraph==0.3.18",
    "langchain-openai==0.3.16",
    "langchain-community==0.3.23",
    "langchain-anthropic",
    "langchain-ollama",
    "python-dotenv",
    "pydantic",
    "tooluniverse",
    "mcp",
    "networkx",
    "faiss-cpu",
    "seaborn",
]

print("Installing missing packages...")
for pkg in _to_install:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q", "--disable-pip-version-check"],
        capture_output=True, text=True
    )
    status = "OK  " if r.returncode == 0 else "FAIL"
    print(f"  [{status}] {pkg}")
    if r.returncode != 0:
        print(f"        {r.stderr[-200:]}")

print("\nDone.")

Installing missing packages...
  [OK  ] fastapi
  [OK  ] uvicorn[standard]
  [OK  ] langchain==0.3.25
  [OK  ] langgraph==0.3.18
  [OK  ] langchain-openai==0.3.16
  [OK  ] langchain-community==0.3.23
  [OK  ] langchain-anthropic
  [OK  ] langchain-ollama
  [OK  ] python-dotenv
  [OK  ] pydantic
  [OK  ] tooluniverse
  [OK  ] mcp
  [OK  ] networkx
  [OK  ] faiss-cpu
  [OK  ] seaborn

Done.


## CELLULE 1 — Configuration de l'environnement

In [2]:
import sys, os

# ── Paths ──────────────────────────────────────────────────────────
GENOMEER_SRC      = "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/src"
TEST_DIR          = "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/tests"
LORA_ADAPTER_PATH = "/mnt/nfs/llmhub/Genomeer/sft/output/Models/v2v4_r64/Apertus-8B-BioInstruct"
BASE_MODEL_ID     = "swiss-ai/Apertus-8B-Instruct-2509"
LOCAL_API_PORT    = 8001
LOCAL_API_URL     = f"http://127.0.0.1:{LOCAL_API_PORT}/v1"
MODEL_ALIAS       = "Apertus-8B-BioInstruct"

# ── Add genomeer source to Python path ────────────────────────────
for p in [GENOMEER_SRC, TEST_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── Environment variables ──────────────────────────────────────────
# HF_HOME intentionally NOT set → transformers uses ~/.cache/huggingface (local NVMe SSD)
os.environ["GENOMEER_RAG_OFFLINE"]       = "1"
os.environ["GENOMEER_SKIP_ENV_INSTALL"]  = "1"

print("Configuration:")
print(f"  Genomeer src  : {GENOMEER_SRC}")
print(f"  Adapter       : {LORA_ADAPTER_PATH}")
print(f"  Base model    : {BASE_MODEL_ID}")
print(f"  HF cache      : ~/.cache/huggingface  (local NVMe, fast)")
print(f"  API URL       : {LOCAL_API_URL}")

# ── Quick imports check ────────────────────────────────────────────
try:
    import torch, transformers, peft
    print(f"\ntorch        : {torch.__version__}  (CUDA: {torch.cuda.is_available()})")
    print(f"transformers : {transformers.__version__}")
    print(f"peft         : {peft.__version__}")
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print(f"GPU          : {props.name}  ({props.total_memory/1e9:.1f} GB VRAM)")
except ImportError as e:
    print(f"MISSING: {e}")

Configuration:
  Genomeer src  : /mnt/nfs/llmhub/MetaLm/sockett/genomeer/src
  Adapter       : /mnt/nfs/llmhub/Genomeer/sft/output/Models/v2v4_r64/Apertus-8B-BioInstruct
  Base model    : swiss-ai/Apertus-8B-Instruct-2509
  HF cache      : ~/.cache/huggingface  (local NVMe, fast)
  API URL       : http://127.0.0.1:8001/v1

torch        : 2.10.0  (CUDA: True)
transformers : 5.4.0
peft         : 0.18.1
GPU          : NVIDIA RTX A5000  (25.4 GB VRAM)


## CELLULE 2 — Démarrage du serveur de modèle (LoRA)

In [3]:
import subprocess, sys, time, os, requests

LOCAL_API_URL  = "http://127.0.0.1:8001/v1"
SERVER_SCRIPT  = "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/tests/serve_lora.py"
PYTHON_BIN     = "/mnt/nfs/llmhub/sft_env/bin/python3"
LOG_FILE       = "/tmp/serve_lora.log"
_server_proc   = None

_already_running = False
try:
    _r = requests.get(f"{LOCAL_API_URL}/models", timeout=2)
    if _r.status_code == 200:
        print(f"Server already running: {_r.json()}")
        _already_running = True
except Exception:
    pass

if not _already_running:
    print(f"Starting model server — logs → {LOG_FILE}")
    print("(~16 GB from local NVMe, ~1 min)\n")

    _logf = open(LOG_FILE, "w")
    _server_proc = subprocess.Popen(
        [PYTHON_BIN, SERVER_SCRIPT],
        stdout=_logf, stderr=_logf,   # FILE not PIPE — avoids 64 KB buffer deadlock
    )
    print(f"PID {_server_proc.pid} — polling every 10 s...")

_ready = _already_running
for _i in range(36):
    if _ready:
        break
    time.sleep(10)
    try:
        with open(LOG_FILE) as _lf:
            _last = [l.rstrip() for l in _lf if l.strip()]
            if _last:
                print(f"  [{(_i+1)*10}s] {_last[-1]}")
    except Exception:
        pass
    try:
        _r = requests.get(f"{LOCAL_API_URL}/models", timeout=3)
        if _r.status_code == 200:
            print(f"\nServer READY  →  {LOCAL_API_URL}")
            print(f"Models: {[m['id'] for m in _r.json()['data']]}")
            _ready = True
    except Exception:
        pass

if not _ready:
    print(f"\nFailed — last 20 lines of {LOG_FILE}:")
    with open(LOG_FILE) as _lf:
        print("".join(_lf.readlines()[-20:]))
    if _server_proc:
        _server_proc.kill()
    raise RuntimeError("Model server did not start")

Starting model server — logs → /tmp/serve_lora.log
(~16 GB from local NVMe, ~1 min)

PID 3517009 — polling every 10 s...
  [20s] INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)

Server READY  →  http://127.0.0.1:8001/v1
Models: ['Apertus-8B-BioInstruct']


## CELLULE 3 — Initialisation du BioAgent

In [4]:
from genomeer.agent.v2 import BioAgent
from uuid import uuid4

agent = BioAgent(
    path="/mnt/nfs/llmhub/MetaLm/sockett/genomeer/tests",
    llm=MODEL_ALIAS,
    source="Custom",
    base_url=LOCAL_API_URL,
    api_key="EMPTY",
    use_tool_retriever=True,
    timeout_seconds=600,
    auto_start_artifacts=True,
    interaction_mode="auto",
    behavior_profile="lora",   # enforces strict output format for fine-tuned models
)

print("BioAgent initialized.")


/mnt/nfs/llmhub/sft_env/lib/python3.10/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer



BioAgent_v1 CONFIGURATION
DEFAULT CONFIG :
  Path: ./data
  Run Dir: .genomeer_runs
  Timeout Seconds: 600
  Llm: ollama/llama3.1
  Temperature: 0.7
  Use Tool Retriever: True

[AGENT LLM] Constructor Override:
  LLM Model: Apertus-8B-BioInstruct
  Source: Custom
  Base URL: http://127.0.0.1:8001/v1

[BioAgent] behavior_profile='lora' applied — stricter format rules active.
[BioAgent] LoRA output normalizer active (profile='lora')
BioAgent initialized.


In [5]:
_ = agent.visualize_graph(mode="manual")

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	qa(qa)
	orchestrator(orchestrator)
	batch_orchestrator(batch_orchestrator)
	input_guard(input_guard)
	generator(generator)
	ensure_env(ensure_env)
	executor(executor)
	validator(validator)
	observer(observer)
	diagnostics(diagnostics)
	finalizer(finalizer)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	finalizer --> __end__;
	qa --> __end__;
	planner -.-> qa;
	planner -.-> orchestrator;
	planner -.-> generator;
	planner -.-> ensure_env;
	planner -.-> input_guard;
	orchestrator -.-> planner;
	orchestrator -.-> input_guard;
	orchestrator -.-> finalizer;
	orchestrator -.-> batch_orchestrator;
	batch_orchestrator -.-> finalizer;
	input_guard -.-> qa;
	input_guard -.-> generator;
	generator -.-> ensure_env;
	generator -.-> qa;
	ensure_env -.-> executor;
	ensure_env -. &nbsp;end&nbsp; .-> __end__;
	executor -.-> validator;
	executor -.-> generator;
	validator -.-> ob

In [6]:
import builtins, re, logging

_real_print = builtins.print

for _noisy in ('uvicorn', 'uvicorn.access', 'uvicorn.error', 'httpx', 'httpcore'):
    logging.getLogger(_noisy).setLevel(logging.CRITICAL)

_LINE = "━" * 60


# ── Message parser ─────────────────────────────────────────────────
def _parse_entry(entry: str):
    lines = entry.split('\n')
    msg_type = 'human' if 'Human Message' in (lines[0] if lines else '') else 'ai'
    body = '\n'.join(l for l in lines if not l.startswith('=')).strip()
    return msg_type, body


def _fmt(msg_type: str, content: str):
    c = content.strip()
    if not c:
        return None

    # PLANNER: route + checklist
    route_m = re.search(r'<next:(\w+)>', c, re.I)
    if route_m:
        route = route_m.group(1).upper()
        steps = [l.strip()[5:].strip() for l in c.splitlines()
                 if l.strip().startswith('- [ ]')]
        out = ['  route → ' + route]
        for i, s in enumerate(steps, 1):
            out.append('    [' + str(i) + '] ' + s[:80])
        return '\n'.join(out)

    # ORCHESTRATOR: running step marker
    run_m = re.search(r'<running\s+step=(\d+)/>', c)
    if run_m:
        desc_m = re.search(r'<description>\s*(.+?)\s*</description>', c, re.S)
        title = desc_m.group(1).strip()[:80] if desc_m else 'Step ' + run_m.group(1)
        title = re.sub(r'^Step\s+\d+[:\.\-]?\s*', '', title)
        return '\n  [' + run_m.group(1) + '] ' + title

    # OBSERVER: step STATUS
    if re.search(r'<STATUS:done>', c, re.I):
        body = re.sub(r'<STATUS:done>', '', c, flags=re.I).strip()
        first = next((l.strip() for l in body.splitlines() if l.strip()), '')
        return '      ✓ done' + (('  —  ' + first[:80]) if first else '')
    if re.search(r'<STATUS:blocked>', c, re.I):
        body = re.sub(r'<STATUS:blocked>', '', c, flags=re.I).strip()
        first = next((l.strip() for l in body.splitlines() if l.strip()), '')
        return '      ✗ BLOCKED' + (('  —  ' + first[:80]) if first else '')

    # INPUT GUARD: OK signal
    if re.search(r'<OK\s*/?\s*>', c, re.I):
        return '      → inputs OK'

    # INPUT GUARD: MISSING inputs
    miss_m = re.search(r'<MISSING>(.*?)</MISSING>', c, re.S | re.I)
    if miss_m:
        items = [x.strip('- ').strip() for x in miss_m.group(1).strip().splitlines() if x.strip()]
        return '      → MISSING: ' + ', '.join(items[:3])

    # GENERATOR: code block
    if re.search(r'<EXECUTE', c, re.I) or re.search(r'^#!(PY|R|BASH|CLI)\b', c, re.M | re.I):
        return '      → code generated'

    # EXECUTOR: execution output (up to 4 lines preview)
    out_m = re.search(r"<observe>Code Execution output:\s*'(.*?)'</observe>", c, re.S | re.I)
    if out_m:
        lines = [l for l in out_m.group(1).strip().splitlines() if l.strip()][:4]
        if lines:
            preview = '\n'.join('         ' + l for l in lines)
            return '      → output:\n' + preview
        return '      → (empty output)'

    # ENV ready — skip silently
    if re.search(r"<observe>Environment '[\w-]+' ready", c, re.I):
        return None

    # Other <observe> — skip
    if c.startswith('<observe>'):
        return None

    # Everything else (verbose INPUT_GUARD text, QA answer, FINALIZER) — skip
    return None


# ── run_agent ──────────────────────────────────────────────────────
def run_agent(prompt, attachments=None, session_id=None):
    _real_print('\n' + _LINE)
    _logs, answer, _err = [], None, None
    builtins.print = lambda *a, **kw: None
    try:
        _logs, answer = agent.go(
            prompt, attachments=attachments or [], session_id=session_id,
        )
    except Exception as _e:
        _err = _e
    finally:
        builtins.print = _real_print
    for entry in (_logs or []):
        mtype, content = _parse_entry(entry)
        line = _fmt(mtype, content)
        if line is not None:
            _real_print(line)
    if _err is not None:
        _real_print('  ✗ ' + type(_err).__name__ + ': ' + str(_err)[:200])
    _real_print(_LINE + '\n')
    if _err is not None:
        raise _err
    return answer


_real_print('run_agent() ready')


run_agent() ready


---
# TESTS
---

## LEVEL 1 — Simple Q&A (pas de code, pas d'outils)
> Vérifie que le modèle route correctement vers le nœud **QA** et répond avec le contexte biologique attendu.

In [7]:
print("=" * 60)
print("TEST 1-A : What is metagenomics?")
print("=" * 60)

sid_1a = str(uuid4())
answer_1a = run_agent(
    "What is metagenomics?",
    session_id=sid_1a,
)
print(answer_1a)


TEST 1-A : What is metagenomics?

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


INFO:     Started server process [3516072]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8910 (Press CTRL+C to quit)


  route → ORCHESTRATOR
    [1] Metagenomics is a technique that sequences all DNA present in a sample rather th

  [1] Metagenomics is a technique that sequences all DNA present in a sample rather th
      → inputs OK
      → code generated
      → output:
         [EXECUTION ERROR] TypeError: run_python_code() got an unexpected keyword argument 'extra_env'
         traceback: Traceback (most recent call last):
           File "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/src/genomeer/agent/v2/BioAgent.py", line 1845, in _executor
             out = run_with_timeout(
      ✗ BLOCKED  —  Python exception detected — execution failed.
      → code generated
      → output:
         [EXECUTION ERROR] TypeError: run_python_code() got an unexpected keyword argument 'extra_env'
         traceback: Traceback (most recent call last):
           File "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/src/genomeer/agent/v2/BioAgent.py", line 1845, in _executor
             out = run_with_timeout(
      ✗ BLOCKE

In [8]:
print("=" * 60)
print("TEST 1-B : 16S vs WGS")
print("=" * 60)

sid_1b = str(uuid4())
answer_1b = run_agent(
    (
        "Compare 16S rRNA amplicon sequencing and whole-genome shotgun (WGS) metagenomics. "
        "List 3 pros and 3 cons for each approach. Do NOT run any code."
    ),
    session_id=sid_1b,
)
print(answer_1b)


TEST 1-B : 16S vs WGS

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INFO:     127.0.0.1:39018 - "POST /api/v1/artifacts/runs/create/e3078482-accc-4fb4-bdaf-65c071117c69 HTTP/1.1" 200 OK
INFO:     127.0.0.1:39034 - "POST /api/v1/artifacts/runs/e3078482-accc-4fb4-bdaf-65c071117c69/publish HTTP/1.1" 200 OK
  route → ORCHESTRATOR
    [1] Pros of 16S rRNA amplicon sequencing over WGS:

  [1] Pros of 16S rRNA amplicon sequencing over WGS
      → inputs OK
      → output:
         [EXECUTION ERROR] TypeError: run_python_code() got an unexpected keyword argument 'extra_env'
         traceback: Traceback (most recent call last):
           File "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/src/genomeer/agent/v2/BioAgent.py", line 1845, in _executor
             out = run_with_timeout(
      ✗ BLOCKED  —  Python exception detected — execution failed.
      → code generated
      → output:
         [EXECUTION ERROR] TypeError: run_python_code() got an unexpected keyword argument 'extra

In [9]:
print("=" * 60)
print("TEST 1-C : Bioinformatics file formats")
print("=" * 60)

sid_1c = str(uuid4())
answer_1c = run_agent(
    (
        "List the 5 most important file formats in metagenomics (FASTQ, FASTA, BAM, SAM, VCF or others). "
        "For each: one-line description, typical file extension, and typical file size range. "
        "Format as a markdown table. No code execution."
    ),
    session_id=sid_1c,
)
print(answer_1c)


TEST 1-C : Bioinformatics file formats

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INFO:     127.0.0.1:37322 - "POST /api/v1/artifacts/runs/create/1c121192-83d2-4eb2-a5cf-84bdddfba641 HTTP/1.1" 200 OK
INFO:     127.0.0.1:37326 - "POST /api/v1/artifacts/runs/1c121192-83d2-4eb2-a5cf-84bdddfba641/publish HTTP/1.1" 200 OK
  route → ORCHESTRATOR
    [1] ### File Format Table

  [1] ### File Format Table
      → inputs OK
      → code generated
      → output:
         [EXECUTION ERROR] TypeError: run_python_code() got an unexpected keyword argument 'extra_env'
         traceback: Traceback (most recent call last):
           File "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/src/genomeer/agent/v2/BioAgent.py", line 1845, in _executor
             out = run_with_timeout(
      ✗ BLOCKED  —  Python exception detected — execution failed.
      → code generated
      → output:
         [EXECUTION ERROR] TypeError: run_python_code() got an unexpected keyword argument 'extra_env'
   

### Level 1 Results
- [ ] TEST 1-A : Definition correcte, langage précis
- [ ] TEST 1-B : Comparaison équilibrée, termes techniques exacts
- [ ] TEST 1-C : Table bien formée, formats bioinformatiques corrects

---
## LEVEL 2 — Q&A avancé (métagénomique approfondie)
> Teste la compréhension des pipelines et des mécanismes biologiques complexes.

In [10]:
print("=" * 60)
print("TEST 2-A : Metagenomics pipeline steps")
print("=" * 60)

sid_2a = str(uuid4())
answer_2a = run_agent(
    (
        "Describe a complete shotgun metagenomics analysis pipeline from raw sequencing reads "
        "to biological interpretation. For each step name the most commonly used tools "
        "(e.g., FastQC, Trimmomatic, MEGAHIT, MetaBAT2, GTDB-Tk, etc.) and explain what "
        "the step produces. No code."
    ),
    session_id=sid_2a,
)
print(answer_2a)


TEST 2-A : Metagenomics pipeline steps

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


[SECURITY BLOCK] Dangerous Python pattern detected: os.system() is forbidden (arbitrary shell execution); use subprocess with a list
Code (first 300 chars): '#!PY\nimport json\nimport csv\nimport subprocess\n\n#!/PY\nimport os\nfrom genomeer.utils.filesystem import get_file, list_files\n# Define run directory\nrun_dir = "/tmp/run-83d4773f-a3ea-4aae-a2fa-e4f85450a703"\n# Define input filenames\nreads_1 = get_file(run_dir, "reads_1.fastq.gz")\nreads_2 = get_file(run_d'
[SECURITY BLOCK] Dangerous Python pattern detected: os.system() is forbidden (arbitrary shell execution); use subprocess with a list
Code (first 300 chars): 'import csv\n\n#!/PY\n# Step 1: FastQC\nimport os\nfrom genomeer.utils.filesystem import get_file\nfrom genomeer.tools.function.fastqc import run_fastqc\nrun_fastqc(get_file(run_dir, "reads_1.fastq.gz"), get_file(run_dir, "reads_2.fastq.gz"), get_file(run_dir, "fastqc_report.html"))\n# Step 2: Trimmomatic\nim'


  route → ORCHESTRATOR
    [1] ### Metagenomic Pipeline Overview

  [1] ### Metagenomic Pipeline Overview
      → inputs OK
      → code generated
      ✗ BLOCKED  —  [SECURITY BLOCK] Code rejected before execution.
      → code generated
      ✗ BLOCKED  —  [SECURITY BLOCK] Code rejected before execution.
      → code generated
      → output:
         [EXECUTION ERROR] TypeError: run_python_code() got an unexpected keyword argument 'extra_env'
         traceback: Traceback (most recent call last):
           File "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/src/genomeer/agent/v2/BioAgent.py", line 1845, in _executor
             out = run_with_timeout(
      ✗ BLOCKED  —  Python exception detected — execution failed.
      → code generated
      → output:
         [EXECUTION ERROR] TypeError: run_python_code() got an unexpected keyword argument 'extra_env'
         traceback: Traceback (most recent call last):
           File "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/src/genomeer/agent/v2

In [11]:
print("=" * 60)
print("TEST 2-B : Assembly metrics")
print("=" * 60)

sid_2b = str(uuid4())
answer_2b = run_agent(
    (
        "Explain the following assembly quality metrics and give typical acceptable ranges "
        "for a metagenome-assembled genome (MAG): N50, L50, total assembly length, "
        "number of contigs, completeness, contamination. "
        "Also explain what BUSCO and CheckM measure and how they differ. No code."
    ),
    session_id=sid_2b,
)
print(answer_2b)


TEST 2-B : Assembly metrics

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INFO:     127.0.0.1:60798 - "POST /api/v1/artifacts/runs/create/4ce30d43-8fd0-4aa3-8756-bcb4eed17e6b HTTP/1.1" 200 OK
INFO:     127.0.0.1:60810 - "POST /api/v1/artifacts/runs/4ce30d43-8fd0-4aa3-8756-bcb4eed17e6b/publish HTTP/1.1" 200 OK
  route → ORCHESTRATOR
    [1] ### Assembly quality metrics

  [1] ### Assembly quality metrics
      → inputs OK
      → code generated
      → output:
         [EXECUTION ERROR] TypeError: run_python_code() got an unexpected keyword argument 'extra_env'
         traceback: Traceback (most recent call last):
           File "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/src/genomeer/agent/v2/BioAgent.py", line 1845, in _executor
             out = run_with_timeout(
      ✗ BLOCKED  —  Python exception detected — execution failed.
      → code generated
      → output:
         [EXECUTION ERROR] TypeError: run_python_code() got an unexpected keyword argument 'extra_env'


### Level 2 Results
- [ ] TEST 2-A : Pipeline complet (QC → Assembly → Binning → Classification → Annotation)
- [ ] TEST 2-B : Métriques correctes avec seuils biologiques (N50 > 50kb, completeness > 90%, contamination < 5%)

---
## LEVEL 3 — Code simple (Python, pas d'outils bioinformatiques)
> L'agent doit générer, exécuter et interpréter du code Python pur.

In [12]:
print("=" * 60)
print("TEST 3-A : Parse FASTA, compute GC content + length")
print("=" * 60)

FASTA_FILE = "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/tests/01-hello-word/sequence_tiny.fasta"

sid_3a = str(uuid4())
answer_3a = run_agent(
    (
        f"Parse the FASTA file at {FASTA_FILE}. "
        "For each sequence compute: total length, GC content (%), "
        "and count of nucleotides A, T, G, C. "
        "Print results as a table: SeqID | Length | GC% | A | T | G | C."
    ),
    attachments=[FASTA_FILE],
    session_id=sid_3a,
)
print("\n>>> ANSWER <<<")
print(answer_3a)


TEST 3-A : Parse FASTA, compute GC content + length

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  route → ORCHESTRATOR
    [1] Parse FASTA file using Biopython SeqIO.parse() to yield SeqRecord objects.
    [2] Loop through SeqRecord objects and extract seq.__len__() and seq.upper().count('
    [3] Loop through SeqRecord objects again and count characters in 'A', 'C', 'G', 'T'.
    [4] Print rows formatted as: SeqID | Length | GC% | A | T | G | C.
    [5] Output results to console.
    [6] Exit script cleanly.

  [1] Parse FASTA file using Biopython SeqIO.parse() to yield SeqRecord objects.
      → inputs OK
      → code generated
      → output:
         [EXECUTION ERROR] TypeError: run_python_code() got an unexpected keyword argument 'extra_env'
         traceback: Traceback (most recent call last):
           File "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/src/genomeer/agent/v2/BioAgent.py", line 1845, in _executor
             out = run_with_timeout(
      ✗ BLOCKED 

In [13]:
print("=" * 60)
print("TEST 3-B : K-mer frequency table (k=2,3,4)")
print("=" * 60)

FASTA_FILE = "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/tests/01-hello-word/sequence_tiny.fasta"

sid_3b = str(uuid4())
answer_3b = run_agent(
    (
        f"Read the FASTA file at {FASTA_FILE}. "
        "Compute k-mer frequencies for k = 2, 3, and 4 on the concatenated sequence (use canonical k-mers). "
        "For each k: report total unique k-mers, top-5 most frequent k-mers, entropy (Shannon). "
        "Save summary to a TSV file and print its path."
    ),
    attachments=[FASTA_FILE],
    session_id=sid_3b,
)
print(answer_3b)


TEST 3-B : K-mer frequency table (k=2,3,4)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


[SECURITY BLOCK] Dangerous Python pattern detected: os.system() is forbidden (arbitrary shell execution); use subprocess with a list
Code (first 300 chars): '#!PY\nimport shutil\nfrom io import StringIO\n\n#!/PY\nimport os\nimport sys\nimport time\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport pandas as pd\nfrom Bio import SeqIO\n\ndef run_python_code(code_str, extra_env=None):\n    """\n    Execute arbitrary Python code within this agent\'s namespace.\n   '
[SECURITY BLOCK] Dangerous bash pattern detected: inline code execution via interpreter -c/-e flag
Script (first 300 chars): '#!BASH\n# Probe 1: Check which Python version is being used\npython --version 2>&1 | head -1\n\n# Probe 2: Check if the environment contains the expected python package\npython -c "import genomeer; print(genomeer.version)"\n\n# Probe 3: Test whether genomeer can be imported in Python\npython <<EOF\nimport ge'
[SECURITY BLOCK] Dangerous bash pattern detected: inline code execution via interprete

  ✗ InternalServerError: Error code: 500 - {'detail': 'CUDA out of memory. Tried to allocate 736.00 MiB. GPU 0 has a total capacity of 23.66 GiB of which 94.88 MiB is free. Including non-PyTorch memory, this process has 22.96
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



InternalServerError: Error code: 500 - {'detail': 'CUDA out of memory. Tried to allocate 736.00 MiB. GPU 0 has a total capacity of 23.66 GiB of which 94.88 MiB is free. Including non-PyTorch memory, this process has 22.96 GiB memory in use. Of the allocated memory 21.51 GiB is allocated by PyTorch, and 1.20 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)'}

In [ ]:
print("=" * 60)
print("TEST 3-C : Visualisation matplotlib")
print("=" * 60)

FASTA_FILE = "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/tests/01-hello-word/sequence_tiny.fasta"
sid_3c = str(uuid4())
answer_3c = run_agent(
    f"Parse {FASTA_FILE}. Generate a 2-panel matplotlib figure: "
    "(1) Bar chart of nucleotide counts (A, T, G, C) per sequence. "
    "(2) Scatter plot of sequence length vs GC content with each sequence labeled. "
    "Save as 'sequence_analysis.png' in the temp dir and provide the download path.",
    attachments=[FASTA_FILE],
    session_id=sid_3c,
)
print(answer_3c)

### Level 3 Results
- [ ] TEST 3-A : Code correct, table lisible, GC% correct
- [ ] TEST 3-B : K-mer canoniques (min(kmer, reverse_complement)), entropy calculée
- [ ] TEST 3-C : Figure sauvegardée, chemin de téléchargement fourni

---
## LEVEL 4 — NCBI + Outils bioinformatiques
> Active les outils du registre Genomeer. Nécessite connexion internet et outils disponibles.

In [ ]:
# Autoriser l'installation d'environnements micromamba pour les tests de niveau 4+
import os
os.environ["GENOMEER_SKIP_ENV_INSTALL"] = "0"
print("GENOMEER_SKIP_ENV_INSTALL = 0  (micromamba auto-install enabled)")

In [ ]:
print("=" * 60)
print("TEST 4-A : NCBI fetch + genome statistics")
print("=" * 60)

sid_4a = str(uuid4())
answer_4a = run_agent(
    "Download the E. coli K-12 MG1655 genome from NCBI using accession NC_000913.3. "
    "Then compute: total assembly length, GC content (%), number of contigs, and N50. "
    "Save results to a text report named 'ecoli_genome_stats.txt'.",
    session_id=sid_4a,
)
print(answer_4a)

In [ ]:
print("=" * 60)
print("TEST 4-B : Two-genome comparison")
print("=" * 60)

sid_4b = str(uuid4())
answer_4b = run_agent(
    "In a single pipeline:\n"
    "1) Download Bacillus subtilis 168 genome (GCF_000009045.1) from NCBI.\n"
    "2) Download E. coli K12 genome (GCF_000005845.2) from NCBI.\n"
    "3) For each genome compute: total length, GC content, number of contigs, N50.\n"
    "4) Create a side-by-side bar chart comparing the two genomes and save as 'genome_comparison.png'.\n"
    "5) Save a summary table to 'comparison_report.txt'.",
    session_id=sid_4b,
)
print(answer_4b)

### Level 4 Results
- [ ] TEST 4-A : Génome téléchargé, stats cohérentes (E.coli ~ 4.6 Mb, GC ~ 50.8%)
- [ ] TEST 4-B : Pipeline multi-étapes, comparaison visuelle générée

---
## LEVEL 5 — Mémoire & Continuité de session
> Vérifie que l'agent conserve le contexte entre les messages d'une même session.

In [ ]:
print("=" * 60)
print("TEST 5-A : Session memory - Part 1")
print("=" * 60)

sid_5 = str(uuid4())   # shared across 5-A, 5-B, 5-C
answer_5a = run_agent(
    "I have a FASTQ file with 1 million paired-end reads (150 bp, Illumina HiSeq). "
    "What quality control steps would you recommend before assembly? "
    "List the steps and the tools you would use.",
    session_id=sid_5,
)
print(answer_5a)

In [ ]:
print("=" * 60)
print("TEST 5-B : Session memory - Part 2")
print("=" * 60)

answer_5b = run_agent(
    "Great. Now which assembly tool would you recommend after those QC steps, and why?",
    session_id=sid_5,
)
print(answer_5b)

In [ ]:
print("=" * 60)
print("TEST 5-C : Session summary recall")
print("=" * 60)

answer_5c = run_agent(
    "Summarize what we discussed in this session.",
    session_id=sid_5,
)
print(answer_5c)

### Level 5 Results
- [ ] TEST 5-A : QC steps corrects (FastQC, Trimmomatic/fastp, adapter trimming, quality filtering)
- [ ] TEST 5-B : Référence au contexte du 5-A (Illumina 150 bp, paired-end, post-QC)
- [ ] TEST 5-C : Résumé cohérent avec toute la session

---
## LEVEL 6 — Pipeline complet métagénomique (test avancé)
> Pipeline multi-étapes complet : QC → Assemblage → Métriques → Rapport

In [ ]:
print("=" * 60)
print("TEST 6 : Full pipeline — NCBI → QC → Assembly → Report")
print("=" * 60)

sid_6 = str(uuid4())
answer_6 = run_agent(
    "Run a complete metagenomics mini-pipeline:\n"
    "1) Download SRA dataset SRR2584857 (E. coli genome, small) from NCBI — use only the first 50,000 reads.\n"
    "2) Run quality control (fastp or trimmomatic) and report: initial reads, reads after QC, "
    "   average quality score, adapter content (%).\n"
    "3) Assemble the QC-filtered reads using MEGAHIT.\n"
    "4) Compute assembly statistics: total length, N50, number of contigs >500 bp, GC content.\n"
    "5) Write a final summary report 'pipeline_report.txt' with all results.",
    session_id=sid_6,
)
print(answer_6)

### Level 6 Results
- [ ] Plan généré avec toutes les étapes dans l'ordre
- [ ] Chaque étape exécutée séquentiellement
- [ ] Observer valide les sorties intermédiaires
- [ ] Rapport final complet et cohérent

---
## RAPPORT FINAL

In [ ]:
print("="*65)
print(f"  MODEL  : {MODEL_ALIAS}")
print(f"  ADAPTER: {LORA_ADAPTER_PATH}")
print("="*65)
print()
print("CHECKLIST — fill manually after reviewing outputs above:")
print()
tests = [
    ("1-A", "Definition simple de la métagénomique"),
    ("1-B", "Comparaison 16S vs WGS"),
    ("1-C", "Table des formats de fichiers"),
    ("2-A", "Pipeline métagénomique complet"),
    ("2-B", "Métriques d'assemblage + BUSCO/CheckM"),
    ("3-A", "Parse FASTA + GC content (Python)"),
    ("3-B", "K-mer frequencies + entropy"),
    ("3-C", "Visualisation matplotlib"),
    ("4-A", "NCBI fetch + genome stats"),
    ("4-B", "Two-genome comparison pipeline"),
    ("5-A", "Session memory: QC recommendation"),
    ("5-B", "Session memory: assembly follow-up"),
    ("5-C", "Session memory: recall"),
    ("6  ", "Full pipeline: SRA → QC → Assembly → Report"),
]
for tid, desc in tests:
    print(f"  [ ] TEST {tid} : {desc}")

In [ ]:
# ── Cleanup : arrêter le serveur si nécessaire ────────────────────
try:
    if _server_proc and _server_proc.poll() is None:
        _server_proc.terminate()
        print("Model server stopped.")
    else:
        print("Server already stopped or was not started here.")
except NameError:
    print("No server process to stop.")